# CMU SCS News: scrape article content



In [ ]:
from pathlib import Path
from time import sleep
import html
import json
import random
import re
from urllib.parse import urlsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm
from urllib3.util.retry import Retry

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATA_DIR = Path("./Data")
LINKS_PATH = DATA_DIR / "CMU SCS AI Articles Links.csv"
OUTPUT_PATH = DATA_DIR / "CMU SCS AI Articles.csv"
ERROR_PATH = DATA_DIR / "CMU SCS AI Article Errors.csv"
DEBUG_DIR = DATA_DIR / "cmu_scs_article_debug"

REQUEST_TIMEOUT = 30
DELAY_RANGE = (0.8, 1.5)
CHECKPOINT_EVERY = 10

# Use 3 for a test run. Change to None for the full corpus.
MAX_ARTICLES = None

SKIP_NON_CMU_DOMAINS = True
RETRY_FAILED_ROWS = True
SAVE_INCOMPLETE_HTML = True

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.cs.cmu.edu/",
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
links = pd.read_csv(LINKS_PATH)
links = links.loc[:, ~links.columns.str.startswith("Unnamed:")]

if "Link" not in links.columns:
    raise ValueError(
        f"{LINKS_PATH} must contain a 'Link' column."
    )

links = (
    links.dropna(subset=["Link"])
    .drop_duplicates(subset=["Link"])
    .reset_index(drop=True)
)

if MAX_ARTICLES is not None:
    links = links.head(MAX_ARTICLES)

print(f"Articles selected for this run: {len(links)}")
display(links.head())

In [ ]:
session = requests.Session()
session.headers.update(HEADERS)

retry_strategy = Retry(
    total=4,
    connect=4,
    read=4,
    status=4,
    backoff_factor=1.0,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    return " ".join(html.unescape(str(value)).split())


def hostname_of(url):
    return urlsplit(url).netloc.lower().split(":")[0]


def is_cmu_domain(url):
    host = hostname_of(url)
    return host == "cmu.edu" or host.endswith(".cmu.edu")


def first_text(soup_or_tag, selectors):
    for selector in selectors:
        node = soup_or_tag.select_one(selector)

        if node:
            value = clean_text(
                node.get_text(" ", strip=True)
            )

            if value:
                return value

    return ""


def first_attribute(soup_or_tag, pairs):
    for selector, attribute in pairs:
        node = soup_or_tag.select_one(selector)

        if node:
            value = clean_text(
                node.get(attribute, "")
            )

            if value:
                return value

    return ""


def iter_json_objects(value):
    if isinstance(value, dict):
        yield value

        for child in value.values():
            yield from iter_json_objects(child)

    elif isinstance(value, list):
        for child in value:
            yield from iter_json_objects(child)


def get_article_json_ld(soup):
    accepted_types = {
        "Article",
        "NewsArticle",
        "ReportageNewsArticle",
        "ScholarlyArticle",
        "BlogPosting",
    }

    for script in soup.select(
        'script[type="application/ld+json"]'
    ):
        raw = script.string or script.get_text()

        if not raw.strip():
            continue

        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            continue

        for obj in iter_json_objects(data):
            obj_type = obj.get("@type", "")
            obj_types = set(
                obj_type
                if isinstance(obj_type, list)
                else [obj_type]
            )

            if obj_types & accepted_types:
                return obj

    return {}


def names_from_json_ld(value):
    if not value:
        return ""

    if isinstance(value, str):
        return clean_text(value)

    if isinstance(value, dict):
        return clean_text(value.get("name", ""))

    if isinstance(value, list):
        names = [
            names_from_json_ld(item)
            for item in value
        ]

        return ", ".join(
            name for name in names if name
        )

    return ""


def normalize_author(value):
    author = clean_text(value)
    author = re.sub(
        r"^\s*(?:By|Author:)\s+",
        "",
        author,
        flags=re.IGNORECASE,
    )
    return author.strip(" /|")


def infer_source(final_url, soup, article_json):
    source = names_from_json_ld(
        article_json.get("publisher")
    )

    if not source:
        source = first_attribute(
            soup,
            [
                ('meta[property="og:site_name"]', "content"),
                ('meta[name="application-name"]', "content"),
            ],
        )

    if source:
        return source

    host = hostname_of(final_url)

    source_map = {
        "www.cs.cmu.edu": (
            "Carnegie Mellon University "
            "School of Computer Science"
        ),
        "www.cmu.edu": "Carnegie Mellon University",
        "ri.cmu.edu": "CMU Robotics Institute",
        "www.ri.cmu.edu": "CMU Robotics Institute",
        "hcii.cmu.edu": (
            "CMU Human-Computer Interaction Institute"
        ),
        "lti.cmu.edu": (
            "CMU Language Technologies Institute"
        ),
        "cylab.cmu.edu": "Carnegie Mellon CyLab",
        "www.cylab.cmu.edu": "Carnegie Mellon CyLab",
        "magazine.cs.cmu.edu": "The LINK Magazine",
    }

    return source_map.get(host, host)

In [ ]:
def element_signature(node):
    if node is None:
        return ""

    classes = ".".join(node.get("class", []))
    identifier = node.get("id", "")

    signature = node.name or "unknown"

    if identifier:
        signature += f"#{identifier}"

    if classes:
        signature += f".{classes}"

    return signature


def has_blocked_ancestor(node):
    blocked_tags = {
        "nav",
        "footer",
        "header",
        "aside",
        "form",
    }

    blocked_fragments = {
        "breadcrumb",
        "sidebar",
        "footer",
        "navigation",
        "related",
        "newsletter",
        "subscribe",
        "social",
        "share",
        "promo",
        "advert",
    }

    for parent in node.parents:
        if getattr(parent, "name", None) in blocked_tags:
            return True

        classes = " ".join(
            parent.get("class", [])
        ).casefold()

        identifier = clean_text(
            parent.get("id", "")
        ).casefold()

        combined = f"{classes} {identifier}"

        if any(
            fragment in combined
            for fragment in blocked_fragments
        ):
            return True

    return False


def candidate_score(node):
    """
    Score a possible article container.

    Long paragraphs are the strongest signal. Navigation lists and repeated
    short labels contribute little.
    """
    if node is None:
        return -1

    paragraph_lengths = []

    for paragraph in node.find_all("p"):
        if has_blocked_ancestor(paragraph):
            continue

        text = clean_text(
            paragraph.get_text(" ", strip=True)
        )

        if len(text) >= 60:
            paragraph_lengths.append(len(text))

    heading_bonus = 250 if node.find("h1") else 0
    paragraph_bonus = 100 * len(paragraph_lengths)

    return (
        sum(paragraph_lengths)
        + paragraph_bonus
        + heading_bonus
    )


def locate_body_container(soup):
    """
    Return:
        container, extraction_method, explicit_body_container
    """
    explicit_selectors = [
        "[itemprop='articleBody']",
        "article .field--name-body",
        "main .field--name-body",
        "article .field--name-field-body",
        "main .field--name-field-body",
        ".node__content .field--type-text-with-summary",
        ".node__content .text-formatted",
        ".field--type-text-with-summary",
        ".field--name-body",
        "article .article-body",
        "article .article__body",
        "article .article-content",
        "article .article__content",
        "article .story-body",
        "article .entry-content",
        "main .article-body",
        "main .article__body",
        "main .article-content",
        "main .story-body",
        "main .entry-content",
        ".news-article-body",
        ".article-body",
        ".article__body",
        ".entry-content",
        ".post-content",
        ".story__body",
        ".story-body",
        ".article-content",
    ]

    for selector in explicit_selectors:
        matches = soup.select(selector)

        if not matches:
            continue

        best = max(
            matches,
            key=candidate_score,
        )

        if candidate_score(best) > 0:
            return best, f"explicit:{selector}", True

    broad_selectors = [
        "#rsRegion-mainContent",
        "#main-content",
        "#main",
        "#content",
        ".region-content",
        ".main-content",
        ".page-content",
        ".content-area",
        ".content-wrapper",
        "article",
        "main",
    ]

    candidates = []

    for selector in broad_selectors:
        for node in soup.select(selector):
            candidates.append(
                (
                    candidate_score(node),
                    selector,
                    node,
                )
            )

    if candidates:
        score, selector, node = max(
            candidates,
            key=lambda item: item[0],
        )

        if score > 0:
            return node, f"broad:{selector}", False

    body = soup.body or soup

    return body, "fallback:body", False


def extract_body(
    soup,
    article_json,
    title="",
    subtitle="",
    author="",
    published_date="",
):
    container, method, explicit_body = locate_body_container(
        soup
    )

    if container is None:
        json_body = clean_text(
            article_json.get("articleBody", "")
        )
        return json_body, "json-ld-only", (
            1 if json_body else 0
        )

    body_soup = BeautifulSoup(
        str(container),
        "html.parser",
    )

    for selector in [
        "script",
        "style",
        "noscript",
        "nav",
        "footer",
        "form",
        "figure",
        "figcaption",
        "aside",
        "svg",
        "button",
        "[class*='share']",
        "[class*='social']",
        "[class*='related']",
        "[class*='trending']",
        "[class*='newsletter']",
        "[class*='subscribe']",
        "[class*='author-bio']",
        "[class*='breadcrumb']",
        "[class*='sidebar']",
        "[class*='media-contact']",
        "[class*='media-inquir']",
        "[id*='media-contact']",
        "[id*='media-inquir']",
    ]:
        for node in body_soup.select(selector):
            node.decompose()

    metadata_values = {
        clean_text(title).casefold(),
        clean_text(subtitle).casefold(),
        clean_text(author).casefold(),
        clean_text(published_date).casefold(),
        (
            f"by {clean_text(author)}".casefold()
            if author
            else ""
        ),
    }

    terminal_headings = {
        "scs news",
        "the breakdown",
        "related stories",
        "related news",
        "related content",
        "you might also like",
        "more from cmu",
        "more from carnegie mellon",
    }

    terminal_prefixes = (
        "related stories",
        "related news",
        "related content",
        "you might also like",
        "more from cmu",
        "more from carnegie mellon",
        "subscribe",
        "newsletter",
    )

    contact_headings = {
        "media inquiry",
        "media inquiries",
        "media contact",
        "media contacts",
        "press contact",
        "press contacts",
    }

    blocks = []
    contact_mode = False

    started = (
        explicit_body
        or body_soup.find("h1") is None
    )

    for node in body_soup.find_all(
        ["h1", "h2", "h3", "h4", "p", "li"]
    ):
        if has_blocked_ancestor(node):
            continue

        text = clean_text(
            node.get_text(" ", strip=True)
        )

        if not text:
            continue

        lowered = text.casefold()

        if node.name == "h1":
            started = True
            continue

        if not started:
            continue

        if lowered in contact_headings:
            contact_mode = True
            continue

        # CMU places contact names and phone numbers before the article body.
        # The first substantial paragraph ends contact mode and is retained.
        if contact_mode:
            if node.name == "p" and len(text) >= 80:
                contact_mode = False
            else:
                continue

        if (
            node.name in {"h2", "h3", "h4"}
            and lowered in terminal_headings
        ):
            break

        if lowered.startswith(terminal_prefixes):
            break

        if lowered in metadata_values:
            continue

        if re.match(
            r"^(?:published|posted|updated)\s+",
            text,
            flags=re.IGNORECASE,
        ):
            continue

        # Suppress isolated contact-like values that survive malformed markup.
        if (
            "@" in text
            and len(text) < 120
        ):
            continue

        if (
            re.fullmatch(
                r"[\d\s()+.-]{7,}",
                text,
            )
            and len(text) < 40
        ):
            continue

        if not blocks or text != blocks[-1]:
            blocks.append(text)

    body = "\n\n".join(blocks).strip()

    if body:
        return body, method, len(blocks)

    json_body = clean_text(
        article_json.get("articleBody", "")
    )

    if json_body:
        return json_body, "json-ld-fallback", 1

    # Last-resort paragraph fallback across the full document.
    fallback_blocks = []
    after_h1 = soup.find("h1") is None

    for node in soup.find_all(["h1", "p"]):
        if has_blocked_ancestor(node):
            continue

        if node.name == "h1":
            after_h1 = True
            continue

        if not after_h1:
            continue

        text = clean_text(
            node.get_text(" ", strip=True)
        )

        if len(text) < 80:
            continue

        lowered = text.casefold()

        if lowered in metadata_values:
            continue

        if not fallback_blocks or text != fallback_blocks[-1]:
            fallback_blocks.append(text)

    fallback_body = "\n\n".join(
        fallback_blocks
    ).strip()

    return (
        fallback_body,
        "fallback:long-paragraphs",
        len(fallback_blocks),
    )


def extract_taxonomy(soup, fragments, selectors):
    area = (
        soup.select_one("article")
        or soup.select_one("main")
        or soup.select_one("#rsRegion-mainContent")
        or soup
    )

    values = []

    for selector in selectors:
        for anchor in area.select(selector):
            value = clean_text(
                anchor.get_text(" ", strip=True)
            )
            if value and value not in values:
                values.append(value)

    for fragment in fragments:
        for anchor in area.select(
            f'a[href*="{fragment}"]'
        ):
            value = clean_text(
                anchor.get_text(" ", strip=True)
            )
            if value and value not in values:
                values.append(value)

    return "; ".join(values)


def debug_stem(url):
    parts = urlsplit(url)

    raw = (
        parts.netloc
        + "_"
        + parts.path.strip("/").replace("/", "_")
    )

    raw = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        raw,
    ).strip("_")

    return raw[:180] or "unknown_page"


def save_debug_bundle(
    raw_html,
    soup,
    url,
):
    stem = debug_stem(url)
    html_path = DEBUG_DIR / f"{stem}.html"
    report_path = DEBUG_DIR / f"{stem}.txt"

    html_path.write_text(
        raw_html,
        encoding="utf-8",
    )

    selector_checks = [
        "[itemprop='articleBody']",
        ".field--name-body",
        ".article-body",
        ".article-content",
        ".entry-content",
        "#rsRegion-mainContent",
        "#main-content",
        "#content",
        "article",
        "main",
        "h1",
        "p",
    ]

    lines = [
        f"URL: {url}",
        f"HTML title: {clean_text(soup.title.get_text()) if soup.title else ''}",
        "",
        "SELECTOR COUNTS",
    ]

    for selector in selector_checks:
        lines.append(
            f"{selector}: {len(soup.select(selector))}"
        )

    lines.extend(
        [
            "",
            "CONTENT CONTAINER CANDIDATES",
        ]
    )

    for selector in [
        "#rsRegion-mainContent",
        "#main-content",
        "#main",
        "#content",
        ".region-content",
        ".main-content",
        ".page-content",
        ".content-area",
        "article",
        "main",
        "body",
    ]:
        for index, node in enumerate(
            soup.select(selector)[:5]
        ):
            lines.append(
                f"{selector}[{index}] "
                f"score={candidate_score(node)} "
                f"signature={element_signature(node)}"
            )

    paragraphs = []

    for paragraph in soup.find_all("p"):
        text = clean_text(
            paragraph.get_text(" ", strip=True)
        )

        if text:
            paragraphs.append(
                (
                    len(text),
                    text[:500],
                    element_signature(
                        paragraph.parent
                    ),
                )
            )

    paragraphs.sort(
        key=lambda item: item[0],
        reverse=True,
    )

    lines.extend(
        [
            "",
            "LONGEST PARAGRAPHS",
        ]
    )

    for length, text, parent in paragraphs[:20]:
        lines.append(
            f"length={length} parent={parent}\n{text}\n"
        )

    report_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )

    return str(html_path), str(report_path)

In [ ]:
def scrape_article_page(
    soup,
    requested_url,
    final_url,
    listing_row,
):
    article_json = get_article_json_ld(soup)

    title = clean_text(
        article_json.get("headline", "")
    )

    if not title:
        title = first_text(
            soup,
            [
                "main h1",
                "article h1",
                "#rsRegion-mainContent h1",
                ".page-title",
                ".entry-title",
                "h1",
            ],
        )

    if not title:
        title = clean_text(
            listing_row.get("Article", "")
        )

    subtitle = first_text(
        soup,
        [
            ".field--name-field-subtitle",
            ".field--name-field-deck",
            ".field--name-field-summary",
            "[class*='dek']",
            "[class*='deck']",
            "[class*='subtitle']",
            "[class*='subhead']",
            "[class*='lead']",
            "[class*='summary']",
            "main h1 + h2",
            "article h1 + h2",
            "#rsRegion-mainContent h1 + h2",
        ],
    )

    if not subtitle:
        subtitle = clean_text(
            article_json.get("description", "")
        )

    if not subtitle:
        subtitle = first_attribute(
            soup,
            [
                ('meta[name="description"]', "content"),
                ('meta[property="og:description"]', "content"),
            ],
        )

    author = names_from_json_ld(
        article_json.get("author")
    )

    if not author:
        author = first_text(
            soup,
            [
                ".field--name-field-author",
                ".field--name-field-byline",
                "[itemprop='author']",
                "[class*='byline']",
                "[class*='author-name']",
                "[rel='author']",
                ".author",
                "main p.author",
                "article p.author",
            ],
        )

    author = normalize_author(author)

    if not author:
        author = clean_text(
            listing_row.get("Listing Author", "")
        )

    published_date = clean_text(
        article_json.get("datePublished", "")
    )

    if not published_date:
        published_date = first_attribute(
            soup,
            [
                (
                    'meta[property="article:published_time"]',
                    "content",
                ),
                ('meta[name="date"]', "content"),
                ('meta[name="publish-date"]', "content"),
                ("time[datetime]", "datetime"),
            ],
        )

    if not published_date:
        published_date = first_text(
            soup,
            [
                ".field--name-field-date",
                ".field--name-created",
                "[class*='publication-date']",
                "[class*='published']",
                "[class*='post-date']",
                ".date",
                "article time",
                "main time",
            ],
        )

    if not published_date:
        published_date = clean_text(
            listing_row.get(
                "Listing Published Date",
                "",
            )
        )

    article_body, body_method, body_block_count = extract_body(
        soup=soup,
        article_json=article_json,
        title=title,
        subtitle=subtitle,
        author=author,
        published_date=published_date,
    )

    topics = extract_taxonomy(
        soup,
        ["/topics/", "/topic/"],
        [
            ".field--name-field-topics a",
            ".field--name-field-topic a",
            ".field--name-field-news-category a",
        ],
    )

    tags = extract_taxonomy(
        soup,
        ["/tags/", "/tag/", "/categories/", "/category/"],
        [
            ".field--name-field-tags a",
            ".field--name-field-keywords a",
        ],
    )

    return {
        "Published Date": published_date,
        "Author": author,
        "Source": infer_source(
            final_url,
            soup,
            article_json,
        ),
        "Article Header": title,
        "Sub_Headings": subtitle,
        "Article Body": article_body,
        "Body Extraction Method": body_method,
        "Body Block Count": body_block_count,
        "Body Character Count": len(article_body),
        "Topics": topics,
        "Tags": tags,
        "Url": requested_url,
        "Final URL": final_url,
        "Listing Published Date": clean_text(
            listing_row.get(
                "Listing Published Date",
                "",
            )
        ),
        "Listing Author": clean_text(
            listing_row.get("Listing Author", "")
        ),
        "Listing Category": clean_text(
            listing_row.get("Listing Category", "")
        ),
        "Listing Type": clean_text(
            listing_row.get("Listing Type", "")
        ),
        "Listing Keywords": clean_text(
            listing_row.get("Listing Keywords", "")
        ),
        "Listing Feeds": clean_text(
            listing_row.get("Listing Feeds", "")
        ),
    }

In [ ]:
def save_checkpoint(records):
    if not records:
        return

    frame = (
        pd.DataFrame(records)
        .drop_duplicates(
            subset="Url",
            keep="last",
        )
        .reset_index(drop=True)
    )

    frame.to_csv(
        OUTPUT_PATH,
        index=False,
        encoding="utf-8-sig",
    )

    frame[
        frame["Error"].fillna("").ne("")
    ].to_csv(
        ERROR_PATH,
        index=False,
        encoding="utf-8-sig",
    )


records = []
completed_urls = set()

if OUTPUT_PATH.exists():
    existing = pd.read_csv(OUTPUT_PATH)
    existing = existing.loc[
        :,
        ~existing.columns.str.startswith("Unnamed:"),
    ]

    if (
        RETRY_FAILED_ROWS
        and "Scrape Status" in existing.columns
    ):
        completed_mask = existing[
            "Scrape Status"
        ].fillna("").isin(
            [
                "Success",
                "Skipped non-CMU domain",
                "Redirected outside CMU; skipped",
            ]
        )
    else:
        completed_mask = pd.Series(
            True,
            index=existing.index,
        )

    completed_urls = set(
        existing.loc[
            completed_mask,
            "Url",
        ]
        .dropna()
        .astype(str)
    )

    records = existing.loc[
        completed_mask
    ].to_dict("records")

    print(
        f"Resume mode: {len(completed_urls)} completed URLs."
    )

In [ ]:
pending_rows = links[
    ~links["Link"].astype(str).isin(completed_urls)
]
pending_records = pending_rows.to_dict("records")

for counter, listing_row in enumerate(
    tqdm(
        pending_records,
        total=len(pending_records),
        desc="CMU AI articles",
    ),
    start=1,
):
    requested_url = str(listing_row["Link"])

    record = {
        "Published Date": clean_text(
            listing_row.get(
                "Listing Published Date",
                "",
            )
        ),
        "Author": clean_text(
            listing_row.get("Listing Author", "")
        ),
        "Source": clean_text(
            listing_row.get(
                "Link Domain",
                hostname_of(requested_url),
            )
        ),
        "Article Header": clean_text(
            listing_row.get("Article", "")
        ),
        "Sub_Headings": "",
        "Article Body": "",
        "Body Extraction Method": "",
        "Body Block Count": 0,
        "Body Character Count": 0,
        "Topics": "",
        "Tags": "",
        "Url": requested_url,
        "Final URL": "",
        "Listing Published Date": clean_text(
            listing_row.get(
                "Listing Published Date",
                "",
            )
        ),
        "Listing Author": clean_text(
            listing_row.get("Listing Author", "")
        ),
        "Listing Category": clean_text(
            listing_row.get("Listing Category", "")
        ),
        "Listing Type": clean_text(
            listing_row.get("Listing Type", "")
        ),
        "Listing Keywords": clean_text(
            listing_row.get("Listing Keywords", "")
        ),
        "Listing Feeds": clean_text(
            listing_row.get("Listing Feeds", "")
        ),
        "Official CMU Domain": is_cmu_domain(
            requested_url
        ),
        "External Redirect": False,
        "HTTP Status": "",
        "Scrape Status": "",
        "Error": "",
        "Debug HTML": "",
        "Debug Report": "",
    }

    if (
        SKIP_NON_CMU_DOMAINS
        and not record["Official CMU Domain"]
    ):
        record["Scrape Status"] = (
            "Skipped non-CMU domain"
        )
        record["Error"] = (
            "Third-party body was not scraped because "
            "SKIP_NON_CMU_DOMAINS=True."
        )
        records.append(record)

        if counter % CHECKPOINT_EVERY == 0:
            save_checkpoint(records)

        continue

    try:
        response = session.get(
            requested_url,
            timeout=REQUEST_TIMEOUT,
            allow_redirects=True,
        )

        record["HTTP Status"] = response.status_code
        record["Final URL"] = response.url
        response.raise_for_status()

        final_is_cmu = is_cmu_domain(response.url)

        record["External Redirect"] = (
            hostname_of(response.url)
            != hostname_of(requested_url)
        )

        if SKIP_NON_CMU_DOMAINS and not final_is_cmu:
            record["Scrape Status"] = (
                "Redirected outside CMU; skipped"
            )
            record["Error"] = (
                "The requested URL redirected outside "
                "cmu.edu."
            )
        else:
            soup = BeautifulSoup(
                response.text,
                "html.parser",
            )

            record.update(
                scrape_article_page(
                    soup=soup,
                    requested_url=requested_url,
                    final_url=response.url,
                    listing_row=listing_row,
                )
            )

            if not record["Article Header"]:
                record["Scrape Status"] = "Incomplete"
                record["Error"] = (
                    "Headline selector returned no value."
                )
            elif not record["Article Body"]:
                record["Scrape Status"] = "Incomplete"
                record["Error"] = (
                    "Article body selector returned no value."
                )
            else:
                record["Scrape Status"] = "Success"

            if (
                SAVE_INCOMPLETE_HTML
                and record["Scrape Status"] == "Incomplete"
            ):
                debug_html, debug_report = save_debug_bundle(
                    raw_html=response.text,
                    soup=soup,
                    url=response.url,
                )

                record["Debug HTML"] = debug_html
                record["Debug Report"] = debug_report

    except requests.RequestException as exc:
        record["Scrape Status"] = "Request error"
        record["Error"] = (
            f"{type(exc).__name__}: {exc}"
        )
    except Exception as exc:
        record["Scrape Status"] = "Parsing error"
        record["Error"] = (
            f"{type(exc).__name__}: {exc}"
        )

    records.append(record)

    if counter % CHECKPOINT_EVERY == 0:
        save_checkpoint(records)

    sleep(random.uniform(*DELAY_RANGE))

save_checkpoint(records)

print(f"Output: {OUTPUT_PATH.resolve()}")
print(f"Error log: {ERROR_PATH.resolve()}")
print(f"Debug directory: {DEBUG_DIR.resolve()}")

In [ ]:
articles = pd.read_csv(OUTPUT_PATH)
articles.head()

In [ ]:
from urllib.parse import urlsplit

articles["Domain"] = articles["Url"].fillna("").apply(
    lambda value: urlsplit(value).netloc
)

status_domain_summary = (
    articles.groupby(
        ["Scrape Status", "Domain"],
        dropna=False,
    )
    .size()
    .rename("Rows")
    .reset_index()
    .sort_values(
        ["Scrape Status", "Rows"],
        ascending=[True, False],
    )
)

display(status_domain_summary)

problem_rows = articles[
    articles["Scrape Status"]
    .fillna("")
    .ne("Success")
]

display(
    problem_rows[
        [
            "Article Header",
            "Domain",
            "Url",
            "Final URL",
            "HTTP Status",
            "Scrape Status",
            "Error",
            "Body Extraction Method",
            "Debug HTML",
            "Debug Report",
        ]
    ].head(50)
)

In [ ]:
print("Duplicate URLs:", articles["Url"].duplicated().sum())

print("\nStatus counts:")
print(
    articles["Scrape Status"]
    .value_counts(dropna=False)
)

print(
    "\nSuccessful rows:",
    articles["Scrape Status"]
    .fillna("")
    .eq("Success")
    .sum(),
)

print(
    "Incomplete rows:",
    articles["Scrape Status"]
    .fillna("")
    .eq("Incomplete")
    .sum(),
)

print(
    "Intentionally skipped external rows:",
    articles["Scrape Status"]
    .fillna("")
    .isin(
        [
            "Skipped non-CMU domain",
            "Redirected outside CMU; skipped",
        ]
    )
    .sum(),
)

print(
    "Request/parsing errors:",
    articles["Scrape Status"]
    .fillna("")
    .isin(
        [
            "Request error",
            "Parsing error",
        ]
    )
    .sum(),
)

print(
    "Missing bodies among successful rows:",
    (
        articles["Scrape Status"]
        .fillna("")
        .eq("Success")
        & articles["Article Body"]
        .fillna("")
        .eq("")
    ).sum(),
)

print("\nBody extraction methods:")
print(
    articles.loc[
        articles["Scrape Status"].eq("Success"),
        "Body Extraction Method",
    ].value_counts(dropna=False)
)